## Pengaturan konfigurasi awal

#### 1. Impor modul yang dibutuhkan

In [1]:
import os
import re
import pandas as pd
from openpyxl import load_workbook

#### 2. Membuat fungsi pendukung

In [2]:
# Membaca data dari file excel
def read_data_from_excel(file_path, sheet_name):
    # Baca sheet dengan header baris 1-3
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=[0,1,2])

    # Gabungkan header multi-level jadi single-level
    df.columns = [
        "_".join([str(c).strip() for c in col if "Unnamed" not in str(c)])
            .lower()
            .replace(" ", "_")
        for col in df.columns.values
    ]

    # Hapus double underscore kalau ada
    df.columns = [c.replace("__", "_").strip("_") for c in df.columns]

    # Cek hasil kolom
    print(df.columns.tolist())

    # Tampilkan data
    print(df.head())

    return df

# Melakukan cleaning data pada dataframe
def cleaning_data(df):
    # bikin copy biar df asli nggak berubah
    df_clean = df.copy()

    # ambil semua kolom object kecuali 'kabupaten/kota'
    object_cols = [col for col in df_clean.select_dtypes(include=["object"]).columns if col != "kabupaten/kota" and col != "provinsi"]

    # convert semua kolom object itu jadi numeric
    # kalau ada "-", "" atau value non-numeric lain -> otomatis jadi NaN
    for col in object_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    # cek hasil konversi
    print(df_clean.dtypes)

    return df_clean

# Mendapatkan data pada periode triwulan berjalan
def get_unpaired_columns(columns):
    # pasangan yang valid
    paired_suffixes = {"ril", "rev", "prov", "kabkot"}

    unpaired = []

    for col in columns:
        # ambil bagian suffix (setelah underscore paling belakang)
        suffix = col.split("_")[-1]

        # cek apakah suffix masuk pasangan
        if suffix not in paired_suffixes:
            unpaired.append(col)

    return unpaired

# Menggabungkan data evaluasi
def concat_evaluasi(list_evaluasi):
    return pd.concat(list_evaluasi, ignore_index=True)    

# Memastikan nama sheet valid
def safe_sheet_name(name):
    return re.sub(r'[:\\/*?\[\]]', "_", str(name))[:31]

# Menulis data ke dalam file excel
def write_to_excel(df, group_col):
    for group_value in df[group_col].unique():
        df_group = df[df[group_col] == group_value]
        kode = str(df_group["kode"].iloc[0])  # ambil kode group_col

        # Nama file sesuai kode group_col
        if group_col == 'provinsi':
            output_file = f"data/output/evaluasi_{kode}.xlsx"
            sheet_name = safe_sheet_name(kode + "00")
        elif group_col == 'kabupaten/kota':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode)
        

        if os.path.exists(output_file):
            # Load workbook lama
            book = load_workbook(output_file)

            if sheet_name in book.sheetnames:
                # Sheet sudah ada → cari baris terakhir
                ws = book[sheet_name]
                startrow = ws.max_row  # tulis setelah baris terakhir

                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=startrow)
                print(f"Data {group_value} ditambahkan ke sheet {sheet_name} di {output_file}")

            else:
                # Sheet belum ada → tambahkan sheet baru
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"Sheet {sheet_name} ditambahkan ke {output_file}")

        else:
            # File belum ada → buat baru
            with pd.ExcelWriter(output_file, engine="openpyxl", mode="w") as writer:
                df_group.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"File baru dibuat: {output_file}, dengan sheet {sheet_name}")


#### 3. Menentukan lokasi file yang akan digunakan sebagai input data

In [3]:
file_path = os.path.join("data/input", "Evaluasi_new.xlsx")

## Evaluasi Diskrepansi Prov vs KabKot

#### 1. Membaca file Evaluasi.xlsx pada sheet Prov vs KabKot

In [66]:
# Membaca data dari file excel
df = read_data_from_excel(file_path, "Diskrepansi Prov vs KabKot")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

['kode', 'provinsi', 'adhb_pdrb_q1-25', 'adhb_pdrb_q2-25', 'adhb_pdrb_q3-25', 'adhb_pkrt_q1-25', 'adhb_pkrt_q2-25', 'adhb_pkrt_q3-25', 'adhb_pklnprt_q1-25', 'adhb_pklnprt_q2-25', 'adhb_pklnprt_q3-25', 'adhb_pkp_q1-25', 'adhb_pkp_q2-25', 'adhb_pkp_q3-25', 'adhb_pmtb_q1-25', 'adhb_pmtb_q2-25', 'adhb_pmtb_q3-25', 'adhb_pi_q1-25', 'adhb_pi_q2-25', 'adhb_pi_q3-25', 'adhb_net_ekspor_q1-25', 'adhb_net_ekspor_q2-25', 'adhb_net_ekspor_q3-25', 'adhk_pdrb_q1-25', 'adhk_pdrb_q2-25', 'adhk_pdrb_q3-25', 'adhk_pkrt_q1-25', 'adhk_pkrt_q2-25', 'adhk_pkrt_q3-25', 'adhk_pklnprt_q1-25', 'adhk_pklnprt_q2-25', 'adhk_pklnprt_q3-25', 'adhk_pkp_q1-25', 'adhk_pkp_q2-25', 'adhk_pkp_q3-25', 'adhk_pmtb_q1-25', 'adhk_pmtb_q2-25', 'adhk_pmtb_q3-25', 'adhk_pi_q1-25', 'adhk_pi_q2-25', 'adhk_pi_q3-25', 'adhk_net_ekspor_q1-25', 'adhk_net_ekspor_q2-25', 'adhk_net_ekspor_q3-25', 'distribusi_pdrb_q1-25', 'distribusi_pdrb_q1-25.1', 'distribusi_pdrb_q2-25', 'distribusi_pdrb_q2-25.1', 'distribusi_pdrb_q3-25', 'distribusi_pdrb

#### 2. Evaluasi diskrepansi ADHB dan ADHK

In [ ]:
# 1. Ambil kolom yang diawali adhb atau adhk
target_cols = [col for col in df_clean.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat list untuk menampung hasil diskrepansi
evaluasi_diskrepansi = []

# 3. Loop tiap kolom target
for col in target_cols:
    for idx, val in df_clean[col].items():
        # PDRB
        if "pdrb" in col:
            # ADHB dan ADHK Q1 & Q2 2025
            if any(q in col for q in ["q1", "q2"]):
                if pd.notna(val) and (val < -5 or val > 5):
                    # Pisahkan nama kolom jadi komponen
                    # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                    parts = col.split("_")
                    
                    periode = parts[-1]  # ambil q1 / q2

                    # tentukan evaluasi berdasarkan isi col
                    if "adhb" in col:
                        evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"
                    elif "adhk" in col:
                        evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"

                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": val,
                        "kolom": col,  # tambahan biar tau sumbernya
                        "evaluasi": evaluasi_text
                    }
                    evaluasi_diskrepansi.append(record)

            # ADHB dan ADHK Q3 2025
            elif "q3" in col:
                if pd.notna(val) and (val < -3 or val > 3):
                    # Pisahkan nama kolom jadi komponen
                    # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                    parts = col.split("_")
                    
                    periode = parts[-1]  # ambil q1 / q2

                    # tentukan evaluasi berdasarkan isi col
                    if "adhb" in col:
                        evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-3%)"
                    elif "adhk" in col:
                        evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-3%)"

                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": val,
                        "kolom": col,  # tambahan biar tau sumbernya
                        "evaluasi": evaluasi_text
                    }
                    evaluasi_diskrepansi.append(record)
        # Komponen
        elif "pdrb" not in col:
            # ADHB dan ADHK Q1, Q2, dan Q3 2025
            if pd.notna(val) and (val < -5 or val > 5):
                # Pisahkan nama kolom jadi komponen
                # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                parts = col.split("_")
                
                periode = parts[-1]  # ambil q1 / q2

                # tentukan evaluasi berdasarkan isi col
                if "adhb" in col:
                    evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"
                elif "adhk" in col:
                    evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"
                    
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "provinsi": df_clean.at[idx, "provinsi"],
                    "periode": periode,
                    "nilai": val,
                    "kolom": col,  # tambahan biar tau sumbernya
                    "evaluasi": evaluasi_text
                }
                evaluasi_diskrepansi.append(record)
                
# 4. Ubah jadi DataFrame biar enak lihat
evaluasi_diskrepansi = pd.DataFrame(evaluasi_diskrepansi)
print(evaluasi_diskrepansi)

    kode           provinsi periode      nilai                  kolom  \
0   14.0               Riau   q2-25   8.000000        adhb_pdrb_q2-25   
1   15.0              Jambi   q2-25  10.000000        adhb_pdrb_q2-25   
2   17.0           Bengkulu   q2-25  42.000000        adhb_pdrb_q2-25   
3   11.0               Aceh   q3-25   4.000000        adhb_pdrb_q3-25   
4   13.0     Sumatera Barat   q3-25   3.010000        adhb_pdrb_q3-25   
..   ...                ...     ...        ...                    ...   
95  61.0   Kalimantan Barat   q2-25  81.003563  adhk_net_ekspor_q2-25   
96  62.0  Kalimantan Tengah   q2-25  -6.039417  adhk_net_ekspor_q2-25   
97  73.0   Sulawesi Selatan   q2-25 -24.002748  adhk_net_ekspor_q2-25   
98  75.0          Gorontalo   q2-25  -8.268608  adhk_net_ekspor_q2-25   
99  81.0             Maluku   q2-25   6.507943  adhk_net_ekspor_q2-25   

                                             evaluasi  
0   Diskrepansi ADHB antara Provinsi dan total Kab...  
1   Diskrep

#### 3. Evaluasi distribusi

In [ ]:
# 1. Ambil semua kolom distribusi
dist_cols = [col for col in df_clean.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in dist_cols:
    base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_dist = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if not c.endswith(".1")][0]
        col1 = [c for c in cols if c.endswith(".1")][0]

        # Ambil periode (contoh: distribusi_pdrb_q1-25 → q1)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Distribusi PDRB & Komponen Q1,Q2,Q3 2025
            if pd.notna(v0) and pd.notna(v1):
                selisih = abs(v0 - v1)
                if selisih > 5:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Diskrepansi Distribusi antara Provinsi dan total Kabupaten/Kota lebih dari (+-5 point)"
                    }
                    evaluasi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_dist = pd.DataFrame(evaluasi_dist)
print(evaluasi_dist)

   kode provinsi periode  nilai  \
0  11.0     Aceh   q1-25   30.0   
1  11.0     Aceh   q1-25    7.0   

                                              kolom  \
0  distribusi_pkrt_q1-25 vs distribusi_pkrt_q1-25.1   
1      distribusi_pi_q1-25 vs distribusi_pi_q1-25.1   

                                            evaluasi  
0  Diskrepansi Distribusi antara Provinsi dan tot...  
1  Diskrepansi Distribusi antara Provinsi dan tot...  


#### 4. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 

In [ ]:
# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 
growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in growth_cols:
    base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
evaluasi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if not c.endswith(".1")][0]
        col1 = [c for c in cols if c.endswith(".1")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]
            
            # Growth Q-to-Q
            if 'q-to-q' in col0 and 'implisit' not in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 0.3:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,3 point)"
                        }
                        evaluasi_growth.append(record)
                # Komponen
                elif pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 0.5 and ('pdrb' not in col0):
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Y-on-Y
            elif "y-on-y" in col0 and "implisit" not in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    # Q1 dan Q2 2025
                    if any(q in col0 for q in ["q1", "q2"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 0.3:
                            print(selisih)
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,3 point)"
                            }
                            evaluasi_growth.append(record)
                    # Q3 2025
                    elif "q3" in col0:
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 0.01:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,01 point)"
                            }
                            evaluasi_growth.append(record)
                # Komponen
                elif pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    # Q1 dan Q2 2025
                    if any(q in col0 for q in ["q1", "q2"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 0.5:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                            }
                            evaluasi_growth.append(record)
                    # Q3 2025
                    elif "q3" in col0:
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 0.05:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,05 point)"
                            }
                            evaluasi_growth.append(record)
                
            # Growth C-to-C
            elif 'c-to-c' in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 0.3:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,3 point)"
                        }
                        evaluasi_growth.append(record)
                # Komponen
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 0.5 and ('pdrb' not in col0):
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Implisit Q-to-Q 
            elif 'implisit_q-to-q' in col0:
                if pd.notna(v0) and pd.notna(v1):
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 0.5:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi Implisit Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Implisit Y-on-Y 
            elif 'implisit_y-on-y' in col0:
                if pd.notna(v0) and pd.notna(v1):
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 0.5:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi Implisit Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                        }
                        evaluasi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_growth = pd.DataFrame(evaluasi_growth)
print(evaluasi_growth)

     kode             provinsi periode     nilai  \
0    32.0           Jawa Barat   q1-25  3.490137   
1    53.0  Nusa Tenggara Timur   q1-25  2.818652   
2    76.0       Sulawesi Barat   q1-25  7.233277   
3    91.0          Papua Barat   q1-25  2.352848   
4    94.0                Papua   q1-25  1.700373   
..    ...                  ...     ...       ...   
158  97.0     Papua Pegunungan   q2-25  2.682524   
159   NaN                  NaN   q2-25  2.212323   
160  13.0       Sumatera Barat   q3-25  1.000000   
161  15.0                Jambi   q3-25  0.600000   
162  16.0     Sumatera Selatan   q3-25  1.000000   

                                                 kolom  \
0    implisit_y-on-y_pdrb_q1-25 vs implisit_y-on-y_...   
1    implisit_y-on-y_pdrb_q1-25 vs implisit_y-on-y_...   
2    implisit_y-on-y_pdrb_q1-25 vs implisit_y-on-y_...   
3    implisit_y-on-y_pdrb_q1-25 vs implisit_y-on-y_...   
4    implisit_y-on-y_pdrb_q1-25 vs implisit_y-on-y_...   
..                         

## Menggabungkan seluruh hasil evaluasi

In [ ]:
evaluasi = concat_evaluasi([evaluasi_diskrepansi, evaluasi_dist, evaluasi_growth])
print(evaluasi.head())

## Menuliskan hasil evaluasi kedalam file excel

In [ ]:
write_to_excel(evaluasi, "provinsi")

## Evaluasi Revisi & Growth

#### 1. Membaca file Evaluasi.xlsx pada sheet Revisi & Growth

In [87]:
# Membaca data dari file excel
df = read_data_from_excel(file_path, "Revisi & Growth")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

['kode', 'kabupaten/kota', 'adhb_pdrb_q1-25_ril', 'adhb_pdrb_q1-25_rev', 'adhb_pdrb_q2-25_ril', 'adhb_pdrb_q2-25_rev', 'adhb_pdrb_q3-25', 'adhb_pkrt_q1-25_ril', 'adhb_pkrt_q1-25_rev', 'adhb_pkrt_q2-25_ril', 'adhb_pkrt_q2-25_rev', 'adhb_pkrt_q3-25', 'adhb_pklnprt_q1-25_ril', 'adhb_pklnprt_q1-25_rev', 'adhb_pklnprt_q2-25_ril', 'adhb_pklnprt_q2-25_rev', 'adhb_pklnprt_q3-25', 'adhb_pkp_q1-25_ril', 'adhb_pkp_q1-25_rev', 'adhb_pkp_q2-25_ril', 'adhb_pkp_q2-25_rev', 'adhb_pkp_q3-25', 'adhb_pmtb_q1-25_ril', 'adhb_pmtb_q1-25_rev', 'adhb_pmtb_q2-25_ril', 'adhb_pmtb_q2-25_rev', 'adhb_pmtb_q3-25', 'adhb_pi_q1-25_ril', 'adhb_pi_q1-25_rev', 'adhb_pi_q2-25_ril', 'adhb_pi_q2-25_rev', 'adhb_pi_q3-25', 'adhb_net_ekspor_q1-25_ril', 'adhb_net_ekspor_q1-25_rev', 'adhb_net_ekspor_q2-25_ril', 'adhb_net_ekspor_q2-25_rev', 'adhb_net_ekspor_q3-25', 'adhk_pdrb_q1-25_ril', 'adhk_pdrb_q1-25_rev', 'adhk_pdrb_q2-25_ril', 'adhk_pdrb_q2-25_rev', 'adhk_pdrb_q3-25', 'adhk_pkrt_q1-25_ril', 'adhk_pkrt_q1-25_rev', 'adhk_pkr

#### 2. Evaluasi revisi ADHB dan ADHK

In [18]:
# 1. Ambil semua kolom ADHB dan ADHK 
revisi_harga_cols = [col for col in df.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_harga_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi ADHB dan ADHK
evaluasi_revisi_harga = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = abs((v1 - v0) / v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                    }
                    evaluasi_revisi_harga.append(record)
                elif selisih > 0.3:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi lebih dari (+-30%) dari nilai rilis"
                    }
                    evaluasi_revisi_harga.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_harga = pd.DataFrame(evaluasi_revisi_harga)
print(evaluasi_revisi_harga)

      kode          kabupaten/kota periode       nilai  \
0   6402.0   Kabupaten Kutai Barat   q1-25  13255000.0   
1   6104.0      Kabupaten Mempawah   q2-25  40000000.0   
2   6271.0      Kota Palangka Raya   q1-25  40000000.0   
3   6107.0       Kabupaten Sintang   q2-25  40000000.0   
4   6212.0  Kabupaten Barito Timur   q1-25  40000000.0   
5   6106.0      Kabupaten Ketapang   q2-25  40000000.0   
6   6102.0    Kabupaten Bengkayang   q1-25  40000000.0   
7   6304.0  Kabupaten Barito Kuala   q2-25  40000000.0   
8   6203.0        Kabupaten Kapuas   q1-25  40000000.0   
9   6108.0   Kabupaten Kapuas Hulu   q2-25  40000000.0   
10  6103.0        Kabupaten Landak   q1-25  40000000.0   
11  6108.0   Kabupaten Kapuas Hulu   q2-25   1000000.0   

                                                kolom  \
0          adhb_pdrb_q1-25_ril vs adhb_pdrb_q1-25_rev   
1          adhb_pdrb_q2-25_ril vs adhb_pdrb_q2-25_rev   
2    adhb_pklnprt_q1-25_ril vs adhb_pklnprt_q1-25_rev   
3    adhb_pklnprt

#### 3. Evaluasi revisi distribusi

In [19]:
# 1. Ambil semua kolom distribusi 
revisi_dist_cols = [col for col in df.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_dist_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_revisi_dist = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            # if pd.notna(v0) and pd.notna(v1) and v1 != 0:
            #     selisih = abs((v1 - v0) / v0)
            #     if v0 * v1 < 0:
            #         record = {
            #             "kode": df.at[idx, "kode"],
            #             "kabupaten/kota": df.at[idx, "kabupaten/kota"],
            #             "periode": periode,
            #             "nilai": v1,
            #             "kolom": col0 + " vs " + col1,
            #             "evaluasi": "Distribusi revisi berlawanan arah"
            #         }
            #         evaluasi_revisi_dist.append(record)
            #     elif selisih > 0.1 and "pi" not in col1 and "net_ekspor" not in col1:
            #         record = {
            #             "kode": df.at[idx, "kode"],
            #             "kabupaten/kota": df.at[idx, "kabupaten/kota"],
            #             "periode": periode,
            #             "nilai": v1,
            #             "kolom": col0 + " vs " + col1,
            #             "evaluasi": "Distribusi revisi lebih dari +- 10 persen"
            #         }
            #         evaluasi_revisi_dist.append(record)

            if pd.notna(v0) and pd.notna(v1):
                selisih = abs(v1 - v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                    }
                    evaluasi_revisi_dist.append(record)
                elif selisih > 5:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi lebih dari (+-5 point) dari nilai rilis"
                    }
                    evaluasi_revisi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_dist = pd.DataFrame(evaluasi_revisi_dist)
print(evaluasi_revisi_dist)

     kode                kabupaten/kota periode  nilai  \
0  9705.0    Kabupaten Mamberamo Tengah   q1-25   6.07   
1  9708.0  Kabupaten Pegunungan Bintang   q2-25  -1.00   

                                               kolom  \
0  distribusi_pklnprt_q1-25_ril vs distribusi_pkl...   
1  distribusi_pklnprt_q2-25_ril vs distribusi_pkl...   

                                            evaluasi  
0  Nilai revisi lebih dari (+-5 point) dari nilai...  
1             Nilai revisi dan nilai rilis beda arah  


#### 4. Evaluasi revisi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 

In [ ]:
object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 
revisi_growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y") or col.startswith("sog_q") or col.startswith("sog_y-on-y")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_growth_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi growth
evaluasi_revisi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
            if ('sog' not in col0 and any(x in col0 for x in ['q-to-q', 'y-on-y', 'c-to-c'])):
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif selisih > 3:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi lebih dari (+-3 point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                # Komponen
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif selisih > 4:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi lebih dari (+-4 point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)

            # SoG PI (Q-to-Q, Y-on-Y)
            elif any(x in col0 for x in ['sog_q', 'sog_y-on-y']):
                if pd.notna(v0) and pd.notna(v1):
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif 'pi' in col0:
                        if v1 < -2 or v1 > 2:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "SoG PI lebih dari +-2 persen"
                            }
                            evaluasi_revisi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_growth = pd.DataFrame(evaluasi_revisi_growth)
print(evaluasi_revisi_growth)

['kabupaten/kota']
       kode         kabupaten/kota periode      nilai  \
0    3305.0      Kabupaten Kebumen   q2-25   3.000000   
1    3306.0    Kabupaten Purworejo   q2-25   7.000000   
2    3316.0        Kabupaten Blora   q1-25  -1.000000   
3    3317.0      Kabupaten Rembang   q1-25   5.000000   
4    3324.0       Kabupaten Kendal   q2-25  -1.000000   
..      ...                    ...     ...        ...   
338  9409.0  Kabupaten Biak Numfor   q2-25  -2.114177   
339  9419.0        Kabupaten Sarmi   q2-25  -4.311966   
340  9420.0       Kabupaten Keerom   q2-25   2.223894   
341  9601.0       Kabupaten Mimika   q2-25 -40.249328   
342  9604.0       Kabupaten Nabire   q2-25   2.160221   

                                                 kolom  \
0       q-to-q_pdrb_q2-25_ril vs q-to-q_pdrb_q2-25_rev   
1       q-to-q_pdrb_q2-25_ril vs q-to-q_pdrb_q2-25_rev   
2       q-to-q_pkrt_q1-25_ril vs q-to-q_pkrt_q1-25_rev   
3       q-to-q_pkrt_q1-25_ril vs q-to-q_pkrt_q1-25_rev   
4     

#### 5. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 

In [ ]:
object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 
value_growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y") or col.startswith("sog_q") or col.startswith("sog_y-on-y")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in value_growth_cols:
    if "prov" in col or "kabkot" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi growth prov vs kabkot dan triwulan berjalan
evaluasi_value_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_prov")][0]
        col1 = [c for c in cols if c.endswith("_kabkot")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
            if ('sog' not in col0 and any(x in col0 for x in ['q-to-q', 'y-on-y', 'c-to-c'])):
                if pd.notna(v0) and pd.notna(v1):
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Beda arah dengan nilai provinsi"
                        }
                        evaluasi_value_growth.append(record)
                    elif selisih > 4:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Selisih dengan nilai provinsi cukup jauh (+-4 point)"
                        }
                        evaluasi_value_growth.append(record)

unpaired_columns = get_unpaired_columns(value_growth_cols)
for col in unpaired_columns:
    for idx, val in df_clean[col].items():
        # PI
        if "pi" in col:
            if pd.notna(val) and (val < -2 or val > 2):
                print(col, idx, val)
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                    "periode": periode,
                    "nilai": val,
                    "kolom": col,
                    "evaluasi": "SoG PI lebih dari +-2 persen"
                }
                evaluasi_value_growth.append(record)

# 4. Jadi DataFrame
evaluasi_value_growth = pd.DataFrame(evaluasi_value_growth)
print(evaluasi_value_growth)

['kabupaten/kota']
['q-to-q_pdrb_q1-25_ril', 'q-to-q_pdrb_q1-25_rev', 'q-to-q_pdrb_q2-25_ril', 'q-to-q_pdrb_q2-25_rev', 'q-to-q_pdrb_q3-25_prov', 'q-to-q_pdrb_q3-25_kabkot', 'q-to-q_pkrt_q1-25_ril', 'q-to-q_pkrt_q1-25_rev', 'q-to-q_pkrt_q2-25_ril', 'q-to-q_pkrt_q2-25_rev', 'q-to-q_pkrt_q3-25_prov', 'q-to-q_pkrt_q3-25_kabkot', 'q-to-q_pklnprt_q1-25_ril', 'q-to-q_pklnprt_q1-25_rev', 'q-to-q_pklnprt_q2-25_ril', 'q-to-q_pklnprt_q2-25_rev', 'q-to-q_pklnprt_q3-25_prov', 'q-to-q_pklnprt_q3-25_kabkot', 'q-to-q_pkp_q1-25_ril', 'q-to-q_pkp_q1-25_rev', 'q-to-q_pkp_q2-25_ril', 'q-to-q_pkp_q2-25_rev', 'q-to-q_pkp_q3-25_prov', 'q-to-q_pkp_q3-25_kabkot', 'q-to-q_pmtb_q1-25_ril', 'q-to-q_pmtb_q1-25_rev', 'q-to-q_pmtb_q2-25_ril', 'q-to-q_pmtb_q2-25_rev', 'q-to-q_pmtb_q3-25_prov', 'q-to-q_pmtb_q3-25_kabkot', 'y-on-y_pdrb_q1-25_ril', 'y-on-y_pdrb_q1-25_rev', 'y-on-y_pdrb_q2-25_ril', 'y-on-y_pdrb_q2-25_rev', 'y-on-y_pdrb_q3-25_prov', 'y-on-y_pdrb_q3-25_kabkot', 'y-on-y_pkrt_q1-25_ril', 'y-on-y_pkrt_q1-25_

In [ ]:
# import re

# def get_unpaired_columns(columns):
#     # pasangan yang valid
#     paired_suffixes = {"ril", "rev", "prov", "kabkot"}

#     unpaired = []

#     for col in columns:
#         # ambil bagian suffix (setelah underscore paling belakang)
#         suffix = col.split("_")[-1]

#         # cek apakah suffix masuk pasangan
#         if suffix not in paired_suffixes:
#             unpaired.append(col)

#     return unpaired

# coba_dulu = get_unpaired_columns(value_growth_cols)
# # print(get_unpaired_columns(value_growth_cols))
# print(coba_dulu)

['sog_q_pdrb_q3-25', 'sog_q_pkrt_q3-25', 'sog_q_pklnprt_q3-25', 'sog_q_pkp_q3-25', 'sog_q_pmtb_q3-25', 'sog_q_pi_q3-25', 'sog_q_net_ekspor_q3-25', 'sog_y-on-y_pdrb_q3-25', 'sog_y-on-y_pkrt_q3-25', 'sog_y-on-y_pklnprt_q3-25', 'sog_y-on-y_pkp_q3-25', 'sog_y-on-y_pmtb_q3-25', 'sog_y-on-y_pi_q3-25', 'sog_y-on-y_net_ekspor_q3-25']


In [ ]:
# print(evaluasi_value_growth)

# # 3. Loop tiap kolom target
# for col in coba_dulu:
#     for idx, val in df_clean[col].items():
#         # PI
#         if "pi" in col:
#             if pd.notna(val) and (val < -2 or val > 2):
#                 print(col, idx, val)
#                 record = {
#                     "kode": df_clean.at[idx, "kode"],
#                     "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
#                     "periode": periode,
#                     "nilai": val,
#                     "kolom": col,
#                     "evaluasi": "SoG PI lebih dari +-2 persen"
#                 }
#                 evaluasi_value_growth.append(record)

# # 4. Jadi DataFrame
# evaluasi_value_growth = pd.DataFrame(evaluasi_value_growth)
# print(evaluasi_value_growth)

[{'kode': np.float64(1104.0), 'kabupaten/kota': 'Kabupaten Aceh Tenggara', 'periode': 'q3-25', 'nilai': np.float64(-1.0), 'kolom': 'q-to-q_pdrb_q3-25_prov vs q-to-q_pdrb_q3-25_kabkot', 'evaluasi': 'Beda arah dengan nilai provinsi'}, {'kode': np.float64(1104.0), 'kabupaten/kota': 'Kabupaten Aceh Tenggara', 'periode': 'q3-25', 'nilai': np.float64(-1.0), 'kolom': 'q-to-q_pmtb_q3-25_prov vs q-to-q_pmtb_q3-25_kabkot', 'evaluasi': 'Beda arah dengan nilai provinsi'}, {'kode': np.float64(1202.0), 'kabupaten/kota': 'Kabupaten Mandailing Natal', 'periode': 'q3-25', 'nilai': np.float64(-1.0), 'kolom': 'y-on-y_pdrb_q3-25_prov vs y-on-y_pdrb_q3-25_kabkot', 'evaluasi': 'Beda arah dengan nilai provinsi'}, {'kode': np.float64(1107.0), 'kabupaten/kota': 'Kabupaten Aceh Barat', 'periode': 'q3-25', 'nilai': np.float64(-1.0), 'kolom': 'y-on-y_pkp_q3-25_prov vs y-on-y_pkp_q3-25_kabkot', 'evaluasi': 'Beda arah dengan nilai provinsi'}, {'kode': np.float64(1102.0), 'kabupaten/kota': 'Kabupaten Aceh Singkil', 

## Menggabungkan seluruh hasil evaluasi

In [ ]:
evaluasi = concat_evaluasi([evaluasi_revisi_harga, evaluasi_revisi_dist, evaluasi_revisi_growth, evaluasi_value_growth])
print(evaluasi.head())

## Menuliskan hasil evaluasi kedalam file excel

In [ ]:
write_to_excel(evaluasi, "kabupaten/kota")

---

## Evaluasi Growth Provinsi vs Kabupaten Kota

#### 1. Membaca file Evaluasi.xlsx

In [52]:
# Membaca data kabupaten kota dari file excel
df_kab = read_data_from_excel(file_path, "Revisi & Growth")

# Membaca data provinsi dari file excel
df_prov = read_data_from_excel(file_path, "Diskrepansi Prov vs KabKot")

['kode', 'kabupaten/kota', 'adhb_pdrb_q1-25_ril', 'adhb_pdrb_q1-25_rev', 'adhb_pdrb_q2-25_ril', 'adhb_pdrb_q2-25_rev', 'adhb_pdrb_q3-25', 'adhb_pkrt_q1-25_ril', 'adhb_pkrt_q1-25_rev', 'adhb_pkrt_q2-25_ril', 'adhb_pkrt_q2-25_rev', 'adhb_pkrt_q3-25', 'adhb_pklnprt_q1-25_ril', 'adhb_pklnprt_q1-25_rev', 'adhb_pklnprt_q2-25_ril', 'adhb_pklnprt_q2-25_rev', 'adhb_pklnprt_q3-25', 'adhb_pkp_q1-25_ril', 'adhb_pkp_q1-25_rev', 'adhb_pkp_q2-25_ril', 'adhb_pkp_q2-25_rev', 'adhb_pkp_q3-25', 'adhb_pmtb_q1-25_ril', 'adhb_pmtb_q1-25_rev', 'adhb_pmtb_q2-25_ril', 'adhb_pmtb_q2-25_rev', 'adhb_pmtb_q3-25', 'adhb_pi_q1-25_ril', 'adhb_pi_q1-25_rev', 'adhb_pi_q2-25_ril', 'adhb_pi_q2-25_rev', 'adhb_pi_q3-25', 'adhb_net_ekspor_q1-25_ril', 'adhb_net_ekspor_q1-25_rev', 'adhb_net_ekspor_q2-25_ril', 'adhb_net_ekspor_q2-25_rev', 'adhb_net_ekspor_q3-25', 'adhk_pdrb_q1-25_ril', 'adhk_pdrb_q1-25_rev', 'adhk_pdrb_q2-25_ril', 'adhk_pdrb_q2-25_rev', 'adhk_pdrb_q3-25', 'adhk_pkrt_q1-25_ril', 'adhk_pkrt_q1-25_rev', 'adhk_pkr

#### 2. Join data provinsi dan kabupaten kota

In [53]:
# Ambil dua digit pertama kode kabupaten → mapping ke provinsi
df_kab["kode_prov"] = df_kab["kode"].astype(str).str[:2].astype(int)

# Join dengan data provinsi
df_final = df_kab.merge(
    df_prov,
    left_on="kode_prov",  # kolom yang dipakai sebagai key di df_kab
    right_on="kode",      # kolom yang dipakai sebagai key di df_prov
    how="left"            # tipe join: left join
)

df_final.info()

ValueError: invalid literal for int() with base 10: 'na'

#### 3. Cleaning data

In [ ]:
# Cleaning data
df_clean = cleaning_data(df_final)
df_clean.info()

#### 4. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 

In [ ]:
object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()
print(object_cols)

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 
# Daftar prefix yang mau diperiksa
prefixes = [
    "q-to-q", 
    "y-on-y", 
    "c-to-c", 
    "implisit_q-to-q", 
    "implisit_y-on-y", 
]

# Menggabungkan prefix jadi pola regex, lalu tambahin syarat _x/_y di akhir
pattern = rf"^({'|'.join(prefixes)}).*(_x|_y)$"

# Filter kolom
revisi_growth_cols = [col for col in df_clean.columns if re.match(pattern, col)]

# 2. Buat base name tanpa suffix `_x/_y`
pairs = {}
for col in revisi_growth_cols:
    # if "ril" in col or "rev" in col:
    if col.endswith("_x") or col.endswith("_y"):
        base = "_".join(col.split("_")[:-1])  # buang suffix _x/_y
        pairs.setdefault(base, []).append(col)
print(pairs)

# 3. Evaluasi growth
evaluasi_revisi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_x")][0]
        col1 = [c for c in cols if c.endswith("_y")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # if pd.notna(v0) and pd.notna(v1) and v0 != 0:
            #     selisih = abs((v1 - v0) / v0)
            #     if v0 * v1 < 0:
            #         record = {
            #             "kode": df_clean.at[idx, "kode_x"],
            #             "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
            #             "periode": periode,
            #             "nilai": v0,
            #             "kolom": col0 + " vs " + col1,
            #             "evaluasi": "Beda arah dengan nilai provinsi"
            #         }
            #         evaluasi_revisi_growth.append(record)
            #     elif selisih > 0.5:
            #         record = {
            #             "kode": df_clean.at[idx, "kode_x"],
            #             "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
            #             "periode": periode,
            #             "nilai": v0,
            #             "kolom": col0 + " vs " + col1,
            #             "evaluasi": "Growth kabupaten kota lebih dari +- 50 persen growth provinsi"
            #         }
            #         evaluasi_revisi_growth.append(record)

            if pd.notna(v0) and pd.notna(v1):
                selisih = abs(v1 - v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df_clean.at[idx, "kode_x"],
                        "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v0,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Beda arah dengan nilai provinsi"
                    }
                    evaluasi_revisi_growth.append(record)
                elif selisih > 4:
                    record = {
                        "kode": df_clean.at[idx, "kode_x"],
                        "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v0,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Selisih dengan nilai provinsi cukup jauh (+-4 point)"
                    }
                    evaluasi_revisi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_growth = pd.DataFrame(evaluasi_revisi_growth)
print(evaluasi_revisi_growth)

## Menggabungkan seluruh hasil evaluasi

In [ ]:
evaluasi = concat_evaluasi([evaluasi_revisi_growth])
print(evaluasi.head())

## Menuliskan hasil evaluasi kedalam file excel

In [ ]:
write_to_excel(evaluasi, "kabupaten/kota")